In [ ]:
import json
data = json.load(open('/kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_qa_data.json'))
json.dump(data[:300], open('/kaggle/working/smoke.json', 'w'))

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [ ]:
%cd /kaggle/working
!cp -r /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/lettucedetect_code/LettuceDetect .
%cd LettuceDetect
!pip install -q -e .

In [4]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python /kaggle/working/LettuceDetect/scripts/train.py \
    --ragtruth-path /kaggle/working/smoke.json \
    --model-name answerdotai/ModernBERT-large \
    --output-dir /kaggle/working/smoke_model \
    --batch-size 2 --epochs 1 --learning-rate 1e-5

Loading weights: 100%|█| 172/172 [00:00<00:00, 1568.06it/s, Materializing param=
ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Starting training on cuda
Training samples: 233, Test samples: 25


Epoch 1/1
Training: 100%|█| 117/117 [02:15<00:00,  1.16s/it, loss=0.0547, avg_loss=0.2466]
Epoch 1 completed in 0:02:15. Average loss: 0.2466

Evaluating...
                                                                                
Detailed Classification Report:
              precision    recall  f1-score   support

   Supported     0.9246    

In [ ]:
%cd /kaggle/working/LettuceDetect
!PYTORCH_ALLOC_CONF=expandable_segments:True python scripts/train.py \
    --ragtruth-path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_qa_data.json \
    --model-name answerdotai/ModernBERT-large \
    --output-dir /kaggle/working/lettucedetect_qa_large \
    --batch-size 2 --epochs 6 --learning-rate 1e-5

/kaggle/working/LettuceDetect
Loading weights: 100%|█| 172/172 [00:00<00:00, 1972.55it/s, Materializing param=
ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Starting training on cuda
Training samples: 4531, Test samples: 503


Epoch 1/6
Training: 100%|█| 2266/2266 [42:02<00:00,  1.11s/it, loss=0.1441, avg_loss=0.173
Epoch 1 completed in 0:42:02. Average loss: 0.1738

Evaluating...
                                                                                
Detailed Classification Report:
              precision    recall  f1-score   supp

In [6]:
!cd /kaggle/working && zip -qr lettucedetect_qa_large.zip lettucedetect_qa_large

In [7]:
!python scripts/evaluate.py --model_path /kaggle/working/lettucedetect_qa_large \
    --data_path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_data.json \
    --evaluation_type example_level
!python scripts/evaluate.py --model_path /kaggle/working/lettucedetect_qa_large \
    --data_path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_data.json \
    --evaluation_type char_level


Evaluating model on test samples: 2700
Loading weights: 100%|█| 174/174 [00:00<00:00, 1986.68it/s, Materializing param=

Task type: Summary

Evaluating model on 900 samples

---- Example-Level Evaluation ----
                                                                                
Detailed Example-Level Classification Report:
              precision    recall  f1-score   support

   Supported     0.8432    0.9037    0.8724       696
Hallucinated     0.5649    0.4265    0.4860       204

    accuracy                         0.7956       900
   macro avg     0.7040    0.6651    0.6792       900
weighted avg     0.7801    0.7956    0.7848       900


Evaluation Results:

Hallucination Detection (Class 1):
  Precision: 0.5649
  Recall: 0.4265
  F1: 0.4860

Supported Content (Class 0):
  Precision: 0.8432
  Recall: 0.9037
  F1: 0.8724

AUROC: 0.7664

Task type: Data2txt

Evaluating model on 900 samples

---- Example-Level Evaluation ----
                                            